04/12/2025 status: better monitoring, filters working, but load is too high (only 3 days processed, kernel dies). Try next: hard code the rules into MySQL and use stored procedures to do filtering procedure

Copy of python script to be used for data insertion

Notes: For the script to work

1. Use Jupyter notebook or any IDE that can support the python libraries --> Be sure you installed the libraries using !pip install (insert library name) --> Ex. !pip install pandas
2. Changes for your individual script are the following:
    -Inserting your password and database name --> in #MySQL Database Connection
    -Near the bottom of the script change the path folder to the folder you are using --> in #Run the script
3. Make sure you create a MYSQL database and table (for the sake of simplicity name your table frisco_traffic).
4. Set up for the SQL Table
    CREATE TABLE IF NOT EXISTS frisco_traffic (
    signalid INT NOT NULL,
    date DATE,
    timestamp INT,         -- Updated to INT for timestamp
    eventcode INT,
    eventparam INT,        -- Updated to INT for eventparam
    --PRIMARY KEY (signalid, timestamp)
); 

In [ ]:
## Create table in MySQL

CREATE TABLE IF NOT EXISTS april_filtered (
    signalid INT NOT NULL,
    date DATE,
    timestamp INT, -- Updated to INT for timestamp
    eventcode INT,
    eventparam INT -- Updated to INT for eventparam
    -- PRIMARY KEY (signalid, timestamp)
);

In [ ]:
DELIMITER //
CREATE PROCEDURE FilterEventsByRules()  -- Removed parameters
BEGIN
    SELECT t.* 
    FROM temp_parquet_chunk t
    JOIN signal_param_rules r ON 
        t.SignalID = r.signalid AND 
        t.EventParam = r.eventparam
    WHERE t.EventCode = 82;
END //
DELIMITER ;


In [12]:
## 1hr 14min for Aug

import os
import pandas as pd
import pyarrow.parquet as pq
import mysql.connector
import threading
import time
from pandas import json_normalize

# MySQL Database Connection
def connect_to_mysql():
    try:
        conn = mysql.connector.connect(
            host="localhost", #Default
            user="root", #Default
            password="minjiminutes", #Insert your MySQL password
            database="frisco_db" #Insert your database name here
        )
        print("✅ Connected to MySQL")
        return conn
    except mysql.connector.Error as e:
        print(f"❌ MySQL Connection Error: {e}")
        return None

# Function to Insert Data in Batches
def batch_insert(cursor, data, mysql_conn):
    if not data:
        print("⚠️ No data to insert.")
        return
    ## Insert table name
    sql_query = """
        INSERT INTO april_filtered (signalid, date, timestamp, eventcode, eventparam)
        VALUES (%s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE 
            date = VALUES(date),
            eventparam = VALUES(eventparam);
    """
    try:
        cursor.executemany(sql_query, data)
        mysql_conn.commit()
        print(f"✅ Inserted {len(data)} rows successfully.")
    except mysql.connector.Error as e:
        print(f"❌ MySQL Insert Error: {e}")
        mysql_conn.rollback()

# Function to Keep MySQL Connection Alive (Ping Every 5 Minutes)
def keep_mysql_alive(mysql_conn):
    while True:
        try:
            mysql_conn.ping(reconnect=True, attempts=3, delay=5)
            print("✅ MySQL connection is still alive.")
        except mysql.connector.Error as e:
            print(f"⚠️ Lost MySQL connection: {e}. Trying to reconnect...")
            mysql_conn = connect_to_mysql()
        time.sleep(300)

# Function to Check MySQL Connection Health
def check_mysql_connection(mysql_conn):
    try:
        mysql_conn.ping(reconnect=True, attempts=3, delay=5)
        return True
    except mysql.connector.Error as e:
        print(f"⚠️ MySQL connection lost: {e}")
        return False

# Function to Read Parquet in Chunks
def read_parquet_in_chunks(file_path, chunk_size=100000):
    """Generator that yields parquet file chunks as DataFrames"""
    try:
        reader = pq.ParquetFile(file_path)
        for batch in reader.iter_batches(batch_size=chunk_size):
            yield batch.to_pandas()
    except Exception as e:
        print(f"❌ Error reading {file_path}: {e}")
        yield pd.DataFrame()  # Return empty DataFrame on error

# CHANGED: New function to filter using stored procedure
def filter_with_stored_procedure(mysql_conn, chunk_df):
    """Filter DataFrame using the MySQL stored procedure"""
    cursor = None
    try:
        cursor = mysql_conn.cursor()
        
        # Create temporary table
        cursor.execute("""
            CREATE TEMPORARY TABLE temp_parquet_chunk (
                SignalID INT,
                Date VARCHAR(10),
                TimestampMs BIGINT,
                EventCode INT,
                EventParam INT
            )
        """)
        
        # Insert data
        insert_data = [
            (row['SignalID'], 
             pd.to_datetime(row['Date']).strftime('%Y-%m-%d'),
             int(row['TimestampMs']),
             int(row['EventCode']),
             int(row['EventParam']))
            for _, row in chunk_df.iterrows()
        ]
        cursor.executemany("""
            INSERT INTO temp_parquet_chunk 
            VALUES (%s, %s, %s, %s, %s)
        """, insert_data)
        
        # Call procedure (now parameterless)
        cursor.callproc("FilterEventsByRules")  # CHANGED
        
        # Get results
        filtered_results = []
        for result in cursor.stored_results():
            filtered_results = result.fetchall()
            
        return pd.DataFrame(filtered_results, 
                          columns=['SignalID','Date','TimestampMs','EventCode','EventParam'])
        
    except mysql.connector.Error as e:
        print(f"❌ Stored procedure error: {e}")
        return pd.DataFrame()
    finally:
        if cursor:
            try:
                cursor.execute("DROP TEMPORARY TABLE IF EXISTS temp_parquet_chunk")
                mysql_conn.commit()
            except Exception as e:
                print(f"⚠️ Cleanup error: {e}")
            cursor.close()  # Ensure cursor always closes

# CHANGED: Modified process_parquet_file function
def process_parquet_file(file_path, mysql_conn, batch_size=10000, chunk_size=100000):
    cursor = mysql_conn.cursor()
    batch_data = []

    print(f"📂 Processing file: {file_path}")

    # Change to track batches with a counter:
    batch_counter = 0  # Add this before the for loop

    for chunk_df in read_parquet_in_chunks(file_path, chunk_size):
        # Ensure MySQL connection is active (unchanged)
        if not check_mysql_connection(mysql_conn):
            print("🔄 Reconnecting to MySQL...")
            mysql_conn = connect_to_mysql()
            if mysql_conn is None:
                print("❌ MySQL reconnect failed. Exiting...")
                return
            cursor = mysql_conn.cursor()

        # Print column names for debugging (unchanged)
        print(f"📋 Columns in {file_path}: {chunk_df.columns.tolist()}")

        # Check if 'EventCode' exists (unchanged)
        if 'EventCode' not in chunk_df.columns:
            print(f"⚠️ 'EventCode' column not found in {file_path}. Skipping file.")
            continue

        # CHANGED: Replace all the dictionary filtering code with this:
        chunk_df['EventCode'] = chunk_df['EventCode'].astype(int, errors='ignore')
        chunk_df_filtered = filter_with_stored_procedure(mysql_conn, chunk_df)
        
        if chunk_df_filtered.empty:
            print("⚠️ No matching rows found, skipping chunk.")
            continue

        print(f"🔍 {len(chunk_df_filtered)} rows match EventCode=82 and signal_param rules")
        
        # Convert filtered results to batch data (unchanged except source)
        for _, row in chunk_df_filtered.iterrows():
            batch_data.append((
                row['SignalID'], 
                row['Date'],
                int(row['TimestampMs']),
                int(row['EventCode']),
                int(row['EventParam'])
            ))

            # Insert data in batches (unchanged)
            # In process_parquet_file(), modify the batch insert section:
            if len(batch_data) >= batch_size:
                print(f"📝 Inserting batch of {len(batch_data)} rows... (Total: {len(batch_data) + batch_counter*batch_size})")
                batch_insert(cursor, batch_data, mysql_conn)
                batch_data.clear()
                batch_counter += 1 ## increment counter

    # Insert any remaining data (unchanged)
    if batch_data:
        print(f"📝 Inserting remaining {len(batch_data)} rows into MySQL...")
        batch_insert(cursor, batch_data, mysql_conn)

    print(f"✅ Completed: {file_path}")

# Function to Process All Parquet Files in a Folder and Subfolders
def process_all_parquet_files(parquet_folder_path):
    mysql_conn = connect_to_mysql()
    if mysql_conn is None:
        print("❌ Cannot connect to MySQL. Exiting...")
        return

    # Keep MySQL connection alive in a separate thread
    threading.Thread(target=keep_mysql_alive, args=(mysql_conn,), daemon=True).start()

    try:
        # Check for Parquet files in the folder and subdirectories
        parquet_files = []
        for root, dirs, files in os.walk(parquet_folder_path):
            for filename in files:
                if filename.endswith(".parquet"):
                    file_path = os.path.join(root, filename)
                    parquet_files.append(file_path)

        print(f"📂 Found {len(parquet_files)} Parquet files to process.")
        
        if not parquet_files:
            print("⚠️ No Parquet files found in the specified folder.")
            return

        # Process files ONE BY ONE with delays
        for i, file_path in enumerate(parquet_files):
            print(f"\n{'='*50}")
            print(f"⏳ Processing file {i+1}/{len(parquet_files)}: {file_path}")
            
            # Process the current file
            process_parquet_file(file_path, mysql_conn)
            
            # Add a delay between files (e.g., 5 seconds)
            time.sleep(5)  
            
            # Optional: Clear memory by forcing garbage collection
            import gc
            gc.collect()
            
            print(f"✅ Finished processing file {i+1}. Waiting before next file...")
        
    except Exception as e:
        print(f"❌ Error: {e}")
    finally:
        mysql_conn.close()
        print("🔒 MySQL connection closed.")

def check_parquet_file_path(folder_path):
    # Example function to process Parquet files
    if os.path.exists(folder_path):
        print(f"Processing files in: {folder_path}")
        # Add your logic to process Parquet files here
    else:
        print(f"Error: The folder path does not exist: {folder_path}")


if __name__ == "__main__":
    # TEST MODE - Single folder
    parquet_folder_path = r"/Users/leiasing/Desktop/ITSS 4395/April Zip/archivedDate=2024-04-30"
    
    print(f"🔍 TEST MODE: Processing single folder\n{'-'*50}")
    print(f"Checking file path: {parquet_folder_path}")
    
    if os.path.exists(parquet_folder_path):
        check_parquet_file_path(parquet_folder_path)
        process_all_parquet_files(parquet_folder_path)  # Will only process this one folder
    else:
        print(f"❌ Error: Folder not found: {parquet_folder_path}")


🔍 TEST MODE: Processing single folder
--------------------------------------------------
Checking file path: /Users/leiasing/Desktop/ITSS 4395/April Zip/archivedDate=2024-04-30
Processing files in: /Users/leiasing/Desktop/ITSS 4395/April Zip/archivedDate=2024-04-30
✅ Connected to MySQL
✅ MySQL connection is still alive.
📂 Found 138 Parquet files to process.

⏳ Processing file 1/138: /Users/leiasing/Desktop/ITSS 4395/April Zip/archivedDate=2024-04-30/290_2024-04-30.parquet
📂 Processing file: /Users/leiasing/Desktop/ITSS 4395/April Zip/archivedDate=2024-04-30/290_2024-04-30.parquet
📋 Columns in /Users/leiasing/Desktop/ITSS 4395/April Zip/archivedDate=2024-04-30/290_2024-04-30.parquet: ['SignalID', 'Date', 'TimestampMs', 'EventCode', 'EventParam']
✅ MySQL connection is still alive.
🔍 9339 rows match EventCode=82 and signal_param rules
📋 Columns in /Users/leiasing/Desktop/ITSS 4395/April Zip/archivedDate=2024-04-30/290_2024-04-30.parquet: ['SignalID', 'Date', 'TimestampMs', 'EventCode', 'E

In [ ]:
## holding space for main function

# (All your existing functions remain the same...)
if __name__ == "__main__":
    
    # Change this to point to the parent "Aug Zip" directory
    parent_folder_path = r"/Users/leiasing/Desktop/ITSS 4395/April Zip"
    
    # Check if the parent folder exists
    if not os.path.exists(parent_folder_path):
        print(f"Error: The folder path does not exist: {parent_folder_path}")
        exit()
    
    # Find all subdirectories that match the archivedDate pattern
    archived_folders = []
    for item in os.listdir(parent_folder_path):
        item_path = os.path.join(parent_folder_path, item)
        if os.path.isdir(item_path) and item.startswith("archivedDate="):
            archived_folders.append(item_path)
    
    if not archived_folders:
        print("⚠️ No archivedDate folders found in the specified directory.")
        exit()
    
    print(f"Found {len(archived_folders)} archivedDate folders to process:")
    for folder in archived_folders:
        print(f" - {folder}")
    
    # Process each archivedDate folder
    for folder_path in archived_folders:
        print(f"\n{'='*50}")
        print(f"Processing folder: {folder_path}")
        print(f"{'='*50}")
        check_parquet_file_path(folder_path)
        process_all_parquet_files(folder_path)

In [ ]:
if __name__ == "__main__":
    # TEST MODE - Single folder
    parquet_folder_path = r"/Users/leiasing/Desktop/ITSS 4395/May Zip/archivedDate=2024-05-15"
    
    print(f"🔍 TEST MODE: Processing single folder\n{'-'*50}")
    print(f"Checking file path: {parquet_folder_path}")
    
    if os.path.exists(parquet_folder_path):
        check_parquet_file_path(parquet_folder_path)
        process_all_parquet_files(parquet_folder_path)  # Will only process this one folder
    else:
        print(f"❌ Error: Folder not found: {parquet_folder_path}")